In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')


In [3]:
# 데이터셋 기본 확인 함수 check() 생성
def check(df, name="Data"):
    print(f"{name} 기본 정보")
    
    print("\n[1] 데이터 크기")
    display(df.shape)
    
    print("\n[2] 컬럼 정보")
    df.info()
    
    print("\n[3] 결측치 개수")
    display(
    df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("결측치 개수")
    )
    
    print("\n[4] 중복 데이터 개수")
    display(df.duplicated().sum())

---

# profile

In [4]:
# 데이터 타입 date형식으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"], format="%Y%m%d")

In [5]:
# 필요없는 Unnamed:0 컬럼 제거
profile = profile.drop('Unnamed: 0', axis=1)

---

# portfolio

In [6]:
# channels마다 파생변수 생성
portfolio['web'] = portfolio['channels'].astype(str).str.contains('web').astype(int)
portfolio['email'] = portfolio['channels'].astype(str).str.contains('email').astype(int)
portfolio['mobile'] = portfolio['channels'].astype(str).str.contains('mobile').astype(int)
portfolio['social'] = portfolio['channels'].astype(str).str.contains('social').astype(int)

# 기존 channels 컬럼 제거
portfolio = portfolio.drop('channels', axis=1)

In [7]:
# portfolio 테이블의 필요없는 인덱스 컬럼 제거
portfolio = portfolio.drop('Unnamed: 0', axis=1)

---

# transcript

In [8]:
# 딕셔너리처럼 생긴 문자열을 진짜 딕셔너리로 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 딕셔너리의 키 -> 새로운 컬럼
value_df = pd.DataFrame(transcript['value'].tolist())
transcript = pd.concat([transcript, value_df], axis=1)

# offer id를 offer_id로 컬럼명 통일
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])

# offer id 컬럼 제거
transcript = transcript.drop('offer id', axis=1)

# value 컬럼 제거
transcript = transcript.drop('value', axis=1)

---

# transcript x profile

In [9]:
# transcript 기준으로 profile 데이터를 left Join
merged_df = pd.merge(
    transcript,
    profile,
    left_on='person',
    right_on='id',
    how='left'
)

# 필요 없는 id 컬럼(person과 중복)은 버리기
merged_df = merged_df.drop(columns='id')

In [10]:
# 결측치 처리
# gender의 결측치 'Unknown'으로 채우기 
merged_df['gender'] = merged_df['gender'].fillna('Unknown')

# age의 118을 결측치(NaN)로 바꿔주기 
merged_df['age'] = merged_df['age'].replace(118, np.nan)

# income은 이미 결측치(NaN) 상태

---

# all_merged

In [11]:
all_merged_df = pd.merge(
    merged_df,
    portfolio,
    left_on='offer_id',
    right_on='id',
    how='left'
)

all_merged_df = all_merged_df.drop(columns='id')

# reward 컬럼명 변경(명확하게)
all_merged_df = all_merged_df.rename(columns={
    "reward_x": "transcript_reward",
    "reward_y": "portfolio_reward"
})

In [12]:
# offer_id 이름 변경 (쿠폰명_difficulty_reward_duration)
portfolio['offer_name'] = (
    portfolio['offer_type'] + '_' + 
    portfolio['difficulty'].astype(str) + '_' + 
    portfolio['reward'].astype(str) + '_' + 
    portfolio['duration'].astype(str)
)
# id : key, offer_name : value
offer_name_dict = portfolio.set_index('id')['offer_name'].to_dict()
all_merged_df['offer_id'] = all_merged_df['offer_id'].map(offer_name_dict)


# 사람(person)별로 먼저 묶고, 그 안에서 시간(time) 순서대로 오름차순 정렬
all_merged_df = all_merged_df.sort_values(by=['person', 'time', 'Unnamed: 0']) # - Unnamed: 0 순서 추가

In [13]:
all_merged_df.to_csv('../data/all_merged.csv', index=False)

In [14]:
# 생성한 all_merged_df를 불러옴 **(새로 추가한 코드)
merged = pd.read_csv('../data/all_merged.csv')


In [15]:
check(merged, 'all_merged.csv')

all_merged.csv 기본 정보

[1] 데이터 크기


(306534, 19)


[2] 컬럼 정보
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         306534 non-null  int64  
 1   person             306534 non-null  object 
 2   event              306534 non-null  object 
 3   time               306534 non-null  int64  
 4   amount             138953 non-null  float64
 5   offer_id           167581 non-null  object 
 6   transcript_reward  33579 non-null   float64
 7   gender             306534 non-null  object 
 8   age                272762 non-null  float64
 9   became_member_on   306534 non-null  object 
 10  income             272762 non-null  float64
 11  portfolio_reward   167581 non-null  float64
 12  difficulty         167581 non-null  float64
 13  duration           167581 non-null  float64
 14  offer_type         167581 non-null  object 
 15  web                167581 non-null  floa

,결측치 개수
transcript_reward,272955
amount,167581
social,138953
duration,138953
mobile,138953
email,138953
offer_id,138953
web,138953
offer_type,138953
portfolio_reward,138953



[4] 중복 데이터 개수


np.int64(0)

---

# promotion

In [16]:
# 조건 1: 쿠폰 타입이 bogo, discount, informational 인 것
cond_offers = merged['offer_type'].isin(['bogo', 'discount', 'informational'])

# 조건 2: 이벤트 종류가 transaction(결제) 인 것
cond_transactions = merged['event'] == 'transaction'

target_df = merged[cond_offers | cond_transactions].copy()

print(target_df['offer_type'].value_counts(dropna=False))
print(target_df['event'].value_counts(dropna=False))
display(target_df.head())
display(target_df.shape)

offer_type
NaN              138953
bogo              71617
discount          69898
informational     26066
Name: count, dtype: int64
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
0,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
1,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
2,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
4,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0


(306534, 19)

In [17]:
# person당 offer_id를 하나의 행으로 설정하여, 흩어진 고객 행동의 순서를 보기 편하게 해주는 "피벗테이블 생성 코드"

# 1. 피벗을 돌릴 '쿠폰 이력서' 데이터만 빼내기
offers_df = target_df[target_df['event'] != 'transaction'].copy()

# 2. 안전한 금고에 보관할 '순수 영수증' 데이터만 빼내기
transactions_df = target_df[target_df['event'] == 'transaction'].copy()

print(f"피벗할 쿠폰 데이터: {len(offers_df)} 줄")
print(f"금고에 보관한 영수증: {len(transactions_df)} 줄")

피벗할 쿠폰 데이터: 167581 줄
금고에 보관한 영수증: 138953 줄


In [18]:
# 1. 시간 순서대로 예쁘게 줄 세우기
offers_df = offers_df.sort_values(['person', 'offer_id', 'time'])

# 2. 'received' 이벤트가 등장할 때마다 1, 아니면 0인 깃발(Flag) 만들기
offers_df['is_received'] = (offers_df['event'] == 'offer received').astype(int)

# 3. 사람과 쿠폰 단위로 묶어서, 깃발을 누적해서 더하기 (Cumsum)
offers_df['offer_cycle'] = offers_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 4. 피벗 돌리기
pivot_df = offers_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle'],
    columns='event',
    values='time',
    aggfunc='min'
).reset_index()

display(pivot_df)


event,person,offer_id,offer_cycle,offer completed,offer received,offer viewed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,414.0,408.0,456.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,528.0,504.0,540.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,576.0,NaN
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,1,NaN,168.0,192.0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,1,NaN,336.0,372.0
...,...,...,...,...,...,...
76272,ffff82501cea40309d5fdd7edcca4a07,discount_10_2_10,1,60.0,0.0,6.0
76273,ffff82501cea40309d5fdd7edcca4a07,discount_10_2_7,1,384.0,336.0,354.0
76274,ffff82501cea40309d5fdd7edcca4a07,discount_10_2_7,2,414.0,408.0,414.0
76275,ffff82501cea40309d5fdd7edcca4a07,discount_10_2_7,3,576.0,576.0,582.0


In [19]:
pivot_df.columns.name = None
pivot_df = pivot_df[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed']]

# # reward 따로 뽑아서 merge
# reward_df = offers_df[offers_df['event'] == 'offer completed'][['person', 'offer_id', 'offer_cycle', 'transcript_reward']]
# pivot_df = pivot_df.merge(reward_df, on=['person', 'offer_id', 'offer_cycle'], how='left')

# portfolio 정보 붙이기
pivot_df = pivot_df.merge(
    portfolio[['offer_name', 'offer_type', 'difficulty', 'reward', 'duration', 'web', 'email', 'mobile', 'social']],
    left_on='offer_id',
    right_on='offer_name',
    how='left'
).drop(columns='offer_name')

# profile 정보 붙이기
pivot_df = pivot_df.merge(
    profile[['id', 'gender', 'age', 'became_member_on', 'income']],
    left_on='person',
    right_on='id',
    how='left'
).drop(columns='id')

display(pivot_df.head())
display(pivot_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,web,email,mobile,social,gender,age,became_member_on,income
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,408.0,456.0,414.0,bogo,5,5,5,1,1,1,1,M,33,2017-04-21,72000.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,504.0,540.0,528.0,discount,10,2,10,1,1,1,1,M,33,2017-04-21,72000.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,NaN,576.0,discount,10,2,7,1,1,1,0,M,33,2017-04-21,72000.0
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,1,168.0,192.0,NaN,informational,0,0,3,0,1,1,1,M,33,2017-04-21,72000.0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,1,336.0,372.0,NaN,informational,0,0,4,1,1,1,0,M,33,2017-04-21,72000.0


(76277, 18)

“행 수 차이 478개는 최종 거래 merge가 아니라 reward_df 붙이는 단계에서 중복 키 때문에 생긴 거였고, 그래서 reward는 따로 붙이지 말고 portfolio의 reward를 쓰는 걸로 정리했습니다.”

In [20]:
# 1. 원본에서 offer_id와 offer_type 짝꿍 사전 만들기
offer_dict = offers_df[['offer_id', 'offer_type']].drop_duplicates().set_index('offer_id')['offer_type'].to_dict()

# 2. 피벗 테이블의 offer_id를 보고, 임시로 쿠폰 타입(bogo, discount)을 가져오기
temp_offer_type = pivot_df['offer_id'].map(offer_dict)

# 3. 기존 숫자였던 'offer_cycle' 컬럼 위에 곧바로 덮어쓰기
pivot_df['offer_cycle'] = temp_offer_type.str.capitalize() + '_' + pivot_df['offer_cycle'].astype(str)

In [21]:
# 피벗테이블에 amount 붙이기

# 1. 금고(transactions_df)에서 영수증 알맹이만 꺼내기
transactions_df = transactions_df[['person', 'time', 'amount']]

# 2. 피벗 테이블(pivot_df)에 영수증(receipts) 1:1 도킹하기
final_df = pivot_df.merge(
    transactions_df,
    left_on=['person', 'offer completed'],
    right_on=['person', 'time'],
    how='left'
).drop(columns='time')

# 3. 태블로 퍼널 시각화를 위한 0/1 파생변수 생성
final_df['is_received'] = final_df['offer received'].notna().astype(int)
final_df['is_viewed'] = final_df['offer viewed'].notna().astype(int)
final_df['is_completed'] = final_df['offer completed'].notna().astype(int)

# 4. 정상 흐름 플래그
final_df['is_normal_flow'] = (
    final_df['offer received'].notna() &
    final_df['offer viewed'].notna() &
    final_df['offer completed'].notna() &
    (final_df['offer received'] <= final_df['offer viewed']) &
    (final_df['offer viewed'] <= final_df['offer completed'])
).astype(int)

# 5. 더블카운팅 처리 플래그
final_df['tx_key'] = final_df['person'].astype(str) + '_' + final_df['offer completed'].astype(str)
final_df['is_valid_view'] = final_df['offer viewed'] <= final_df['offer completed']

final_df = final_df.sort_values(
    by=['tx_key', 'is_valid_view', 'offer viewed'],
    ascending=[True, False, False]
)

final_df['primary_flag'] = ~final_df.duplicated(subset=['tx_key'], keep='first')
final_df['is_deduplicated'] = (final_df['primary_flag'] & final_df['is_valid_view']).astype(int)

promo_total_revenue = final_df.loc[final_df['is_deduplicated'] == 1, 'amount'].sum()

final_df = final_df.drop(columns=['tx_key', 'is_valid_view', 'primary_flag'])
# print(f"[최종] 프로모션 귀속 총매출액: ${promo_total_revenue:,.2f}")
# print(f"[최종] 귀속된 거래 수: {final_df['is_deduplicated'].sum():,}")
display(final_df.head())
display(final_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,M,33,2017-04-21,72000.0,8.57,1,1,1,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,M,33,2017-04-21,72000.0,14.11,1,1,1,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,M,33,2017-04-21,72000.0,10.27,1,0,1,0,0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,informational,0,0,4,...,M,33,2017-04-21,72000.0,NaN,1,1,0,0,0
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,informational,0,0,3,...,M,33,2017-04-21,72000.0,NaN,1,1,0,0,0


(76277, 24)

In [22]:
print(f"전체 행 수: {len(final_df):,}")
print(f"정상 흐름(is_normal_flow) 행 수: {final_df['is_normal_flow'].sum():,}")
print(f"더블카운팅 제거 후(is_deduplicated) 행 수: {final_df['is_deduplicated'].sum():,}")
print(f"차이: {(final_df['is_normal_flow'].sum() - final_df['is_deduplicated'].sum()):,}")

전체 행 수: 76,277
정상 흐름(is_normal_flow) 행 수: 23,267
더블카운팅 제거 후(is_deduplicated) 행 수: 22,333
차이: 934


In [23]:
# # 1) 두 조건의 개수 확인
# count_true_conversion = promotion['is_normal_flow'].sum()
# count_valid_attr = promotion['is_deduplicated'].sum()
# diff = count_true_conversion - count_valid_attr

# print(f"is_true_conversion 합계: {count_true_conversion:,}")
# print(f"valid_attribution_flag 합계: {count_valid_attr:,}")
# print(f"차이: {diff:,}")

# # 2) is_true_conversion=1 이지만 valid_attribution_flag=0 인 행만 추출
# # - last touch로 선택되지 못한 값들 선택
# lost_rows = promotion[
#     (promotion['is_normal_flow'] == 1) &
#     (promotion['is_deduplicated'] == 0)
# ].copy()

# print(f"전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: {len(lost_rows):,}")

# # 3) 검증
# assert len(lost_rows) == diff, "차이 수와 실제 빠진 행 수가 다름"

# display(lost_rows.head())

In [24]:
final_df.to_csv('../data/promotion_df.csv', index=False)

csv저장 완료하고 데이터 로드부터 시작

In [25]:
# 프로모션 데이터 불러오기 (새로 추가된 코드)
promotion = pd.read_csv('../data/promotion_df.csv')
merged = pd.read_csv('../data/all_merged.csv')


In [26]:
check(promotion, 'promotiom_df.csv')

promotiom_df.csv 기본 정보

[1] 데이터 크기


(76277, 24)


[2] 컬럼 정보
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76277 entries, 0 to 76276
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   person            76277 non-null  object 
 1   offer_id          76277 non-null  object 
 2   offer_cycle       76277 non-null  object 
 3   offer received    76277 non-null  float64
 4   offer viewed      57725 non-null  float64
 5   offer completed   33101 non-null  float64
 6   offer_type        76277 non-null  object 
 7   difficulty        76277 non-null  int64  
 8   reward            76277 non-null  int64  
 9   duration          76277 non-null  int64  
 10  web               76277 non-null  int64  
 11  email             76277 non-null  int64  
 12  mobile            76277 non-null  int64  
 13  social            76277 non-null  int64  
 14  gender            66501 non-null  object 
 15  age               76277 non-null  int64  
 16  became_member_on  76277 non-n

,결측치 개수
offer completed,43176
amount,43176
offer viewed,18552
income,9776
gender,9776
person,0
social,0
is_normal_flow,0
is_completed,0
is_viewed,0



[4] 중복 데이터 개수


np.int64(0)

아마 transactions_df는 따로 없는 df라 따로 저장을 해야하나??
merged 를 사용하면 될 듯

In [27]:
# 이번 프로모션 모든 매출의 합(총매출액)
total_revenue = merged['amount'].sum()
print(f"총 매출액: {total_revenue:,}$")

# transactions_df가 실제 결제(transaction)만 모아둔 표이고, 그 안의 amount가 각 결제 금액임

총 매출액: 1,775,451.97$


In [28]:
# 최종 인정된 행들의 amount만 합쳐서 프로모션 귀속 총매출액 계산
print(f"[최종] 프로모션 귀속 총매출액: ${promo_total_revenue:,.2f}")
print(f"[최종] 귀속된 거래 수: {promotion['is_deduplicated'].sum():,}")
count_true_conversion = promotion['is_normal_flow'].sum()
print(f"is_normal_flow 합계: {count_true_conversion:,}")

[최종] 프로모션 귀속 총매출액: $442,993.24
[최종] 귀속된 거래 수: 22,333
is_normal_flow 합계: 23,267


In [29]:
# profit(순수익) 구하기
# profit 계산하기! (amount - reward_cost)
promotion['profit'] = promotion['amount'] - promotion['reward']
display(promotion.head(10))

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,profit
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,33,2017-04-21,72000.0,8.57,1,1,1,0,0,3.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,33,2017-04-21,72000.0,14.11,1,1,1,0,0,12.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,33,2017-04-21,72000.0,10.27,1,0,1,0,0,8.27
3,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,informational,0,0,4,...,33,2017-04-21,72000.0,NaN,1,1,0,0,0,NaN
4,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,informational,0,0,3,...,33,2017-04-21,72000.0,NaN,1,1,0,0,0,NaN
5,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,bogo,5,5,5,...,118,2018-04-25,NaN,NaN,1,1,0,0,0,NaN
6,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,bogo,5,5,5,...,118,2018-04-25,NaN,NaN,1,1,0,0,0,NaN
7,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,40,2018-01-09,57000.0,11.93,1,1,1,1,1,8.93
8,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,40,2018-01-09,57000.0,22.05,1,1,1,1,1,17.05
9,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,discount,20,5,10,...,40,2018-01-09,57000.0,22.05,1,1,1,1,0,17.05


In [30]:
# 실제 reward 총지출액 (지출된 총 지출 비용)
total_reward_cost = promotion.loc[promotion['offer completed'].notna(), 'reward'].sum()

# ROI = 프로모션 투자(reward) 대비 수익(프로모션 총매출액)률
roi = (promo_total_revenue - total_reward_cost) / total_reward_cost * 100

print(f"프로모션 귀속 총 매출액: ${promo_total_revenue:,.2f}")
print(f"실제 reward 총지출액: ${total_reward_cost:,.2f}")
print(f"ROI: {roi:.2f}%")
print(f"실제 수익: {promo_total_revenue - total_reward_cost}")


프로모션 귀속 총 매출액: $442,993.24
실제 reward 총지출액: $162,426.00
ROI: 172.74%
실제 수익: 280567.24


In [31]:
# 3. [유효 지출] 진짜 마케팅 퍼널(received->viewed->trans->comp)을 제대로 밟은 고객에게 쓴 돈
valid_reward_cost = promotion.loc[promotion['offer completed'].notna() & promotion['is_deduplicated'], 'reward'].sum()

# 4. [낭비된 지출] 체리피커나 더블카운팅으로 허공에 날린 돈
wasted_reward_cost = total_reward_cost - valid_reward_cost

print(f"💸 [재무 기준] 프로모션 총 리워드 비용: ${total_reward_cost:,.2f}")
print("-" * 60)
print(f"🎯 [마케팅 기준] 광고로 유도된 '유효 지출': ${valid_reward_cost:,.2f} ({valid_reward_cost/total_reward_cost*100:.1f}%)")
print(f"🗑️ [개선 필요] 체리피커 등에게 '낭비된 지출': ${wasted_reward_cost:,.2f} ({wasted_reward_cost/total_reward_cost*100:.1f}%)")

💸 [재무 기준] 프로모션 총 리워드 비용: $162,426.00
------------------------------------------------------------
🎯 [마케팅 기준] 광고로 유도된 '유효 지출': $109,742.00 (67.6%)
🗑️ [개선 필요] 체리피커 등에게 '낭비된 지출': $52,684.00 (32.4%)


In [32]:
# 1. 분석을 위해 '순수 프로모션 기여 데이터'만 따로 떼어냅니다.
valid_promo_df = promotion[promotion['is_deduplicated'] == 1].copy()

print("[스타벅스 프로모션 심층 분석 리포트]\n")

# ==========================================
# 1️⃣ 프로모션 객단가 (AOV) 분석
# ==========================================
# 전체 결제 건의 객단가 vs 프로모션 결제 건의 객단가
total_aov = merged['amount'].mean()
promo_aov = valid_promo_df['amount'].mean()

print("1. 장바구니 크기(AOV) 효과")
print(f"   - 평소 평균 결제액 (자연 매출 객단가): ${total_aov:.2f}")
print(f"   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): ${promo_aov:.2f}")
if promo_aov > total_aov:
    print(f"인사이트: 쿠폰을 주면 고객들이 평소보다 ${promo_aov - total_aov:.2f} 만큼 더 씁니다! (업셀링 성공)\n")
else:
    print(f"인사이트: 쿠폰을 줘도 결제액이 늘지 않습니다. 구매 조건(허들)을 높여야 합니다.\n")

# ==========================================
# 2️⃣ BOGO vs Discount 맞짱 뜨기 (A/B Test)
# ==========================================
print("2. 프로모션 타입별 성과 (BOGO vs Discount)")
# 타입별로 묶어서 결제 건수와 매출액 합산
type_performance = valid_promo_df.groupby('offer_id').agg(
    결제건수=('person', 'count'),
    기여매출액=('amount', 'sum')
).reset_index()

for index, row in type_performance.iterrows():
    print(f"   - [{row['offer_id'].upper()}] 기여 결제: {row['결제건수']:,}건 / 기여 매출: ${row['기여매출액']:,.2f}")
print("인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.\n")


[스타벅스 프로모션 심층 분석 리포트]

1. 장바구니 크기(AOV) 효과
   - 평소 평균 결제액 (자연 매출 객단가): $12.78
   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): $19.84
인사이트: 쿠폰을 주면 고객들이 평소보다 $7.06 만큼 더 씁니다! (업셀링 성공)

2. 프로모션 타입별 성과 (BOGO vs Discount)
   - [BOGO_10_10_5] 기여 결제: 2,654건 / 기여 매출: $62,602.39
   - [BOGO_10_10_7] 기여 결제: 2,467건 / 기여 매출: $58,204.91
   - [BOGO_5_5_5] 기여 결제: 3,448건 / 기여 매출: $68,251.35
   - [BOGO_5_5_7] 기여 결제: 2,031건 / 기여 매출: $35,512.63
   - [DISCOUNT_10_2_10] 기여 결제: 4,375건 / 기여 매출: $78,095.81
   - [DISCOUNT_10_2_7] 기여 결제: 1,981건 / 기여 매출: $39,576.25
   - [DISCOUNT_20_5_10] 기여 결제: 1,147건 / 기여 매출: $30,280.09
   - [DISCOUNT_7_3_7] 기여 결제: 4,230건 / 기여 매출: $70,469.81
인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.



In [33]:
# 기존 결제건수=('tx_key', 'count')였던 걸
# 기존 결제건수=('person', 'count')으로 바꿔도 되는 것은
# count는 값의 종류가 아니라 비어 있지 않은 행 수를 세기 때문이다.

In [34]:
target_df

,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
0,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
1,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
2,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
4,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306529,258361,ffff82501cea40309d5fdd7edcca4a07,transaction,576,14.23,NaN,NaN,F,45.0,2016-11-25,62000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306530,258362,ffff82501cea40309d5fdd7edcca4a07,offer completed,576,NaN,discount_10_2_7,2.0,F,45.0,2016-11-25,62000.0,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0
306531,262475,ffff82501cea40309d5fdd7edcca4a07,offer viewed,582,NaN,discount_10_2_7,NaN,F,45.0,2016-11-25,62000.0,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0
306532,274809,ffff82501cea40309d5fdd7edcca4a07,transaction,606,10.12,NaN,NaN,F,45.0,2016-11-25,62000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# 쿠폰별 이탈률도 확인
offer_list = sorted(merged['offer_id'].dropna().unique())

for offer in offer_list:
    # 1) 해당 쿠폰의 received / viewed / completed만 모으기
    offer_raw_df = merged[merged['offer_id'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])

    # 2) promotion에서 해당 쿠폰만 필터링
    offer_final_df = promotion[promotion['offer_id'] == offer].copy()

    # 3) 조회 후 완료된 건수
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    # 4) 전환율 / 이탈률
    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    print(f"=== {offer} 퍼널 전환율 및 이탈률 ===")
    print(f"1단계 (발송): {step1_received:,}건")
    print(f"2단계 (조회): {step2_viewed:,}건 (전환율: {view_rate:.1f}%, 이탈률: {view_drop_rate:.1f}%)")
    print(f"3단계 (결제): {step3_completed:,}건 (전환율: {purchase_rate:.1f}%, 이탈률: {purchase_drop_rate:.1f}%)")
    print(f"최종 전환율: {total_cvr:.1f}%")
    print("-" * 50)


=== bogo_10_10_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,593건
2단계 (조회): 7,298건 (전환율: 96.1%, 이탈률: 3.9%)
3단계 (결제): 2,739건 (전환율: 37.5%, 이탈률: 62.5%)
최종 전환율: 36.1%
--------------------------------------------------
=== bogo_10_10_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,658건
2단계 (조회): 6,716건 (전환율: 87.7%, 이탈률: 12.3%)
3단계 (결제): 2,582건 (전환율: 38.4%, 이탈률: 61.6%)
최종 전환율: 33.7%
--------------------------------------------------
=== bogo_5_5_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,571건
2단계 (조회): 7,264건 (전환율: 95.9%, 이탈률: 4.1%)
3단계 (결제): 3,514건 (전환율: 48.4%, 이탈률: 51.6%)
최종 전환율: 46.4%
--------------------------------------------------
=== bogo_5_5_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,677건
2단계 (조회): 4,171건 (전환율: 54.3%, 이탈률: 45.7%)
3단계 (결제): 2,106건 (전환율: 50.5%, 이탈률: 49.5%)
최종 전환율: 27.4%
--------------------------------------------------
=== discount_10_2_10 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,597건
2단계 (조회): 7,327건 (전환율: 96.4%, 이탈률: 3.6%)
3단계 (결제): 4,576건 (전환율: 62.5%, 이탈률: 37.5%)
최종 전환율: 60.2%
--------------------------------------------------
===

In [36]:
rows = []

for offer in offer_list:
    offer_raw_df = merged[merged['offer_id'] == offer].copy()
    offer_final_df = promotion[promotion['offer_id'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    rows.append({
        'offer_name': offer,
        'received': step1_received,
        'viewed': step2_viewed,
        'completed': step3_completed,
        'view_rate(%)': round(view_rate, 1),
        'view_drop_rate(%)': round(view_drop_rate, 1),
        'purchase_rate(%)': round(purchase_rate, 1),
        'purchase_drop_rate(%)': round(purchase_drop_rate, 1),
        'total_cvr(%)': round(total_cvr, 1),
    })

offer_funnel_df = pd.DataFrame(rows)

# 최종 전환율 내림차순으로 정렬해서 표시
display(offer_funnel_df.sort_values('total_cvr(%)', ascending=False))


,offer_name,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
4,discount_10_2_10,7597,7327,4576,96.4,3.6,62.5,37.5,60.2
7,discount_7_3_7,7646,7337,4348,96.0,4.0,59.3,40.7,56.9
2,bogo_5_5_5,7571,7264,3514,95.9,4.1,48.4,51.6,46.4
0,bogo_10_10_5,7593,7298,2739,96.1,3.9,37.5,62.5,36.1
1,bogo_10_10_7,7658,6716,2582,87.7,12.3,38.4,61.6,33.7
5,discount_10_2_7,7632,4118,2102,54.0,46.0,51.0,49.0,27.5
3,bogo_5_5_7,7677,4171,2106,54.3,45.7,50.5,49.5,27.4
6,discount_20_5_10,7668,2663,1300,34.7,65.3,48.8,51.2,17.0
8,informational_0_0_3,7618,6687,0,87.8,12.2,0.0,100.0,0.0
9,informational_0_0_4,7617,4144,0,54.4,45.6,0.0,100.0,0.0


In [37]:
# 고객 세그먼트별 전환/이탈률 보기



In [38]:
# 세그먼트 분석용 데이터 복사 (bogo랑 discount만!)
seg_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

# 공통 플래그 만들기
seg_df['viewed_flag'] = seg_df['offer viewed'].notna().astype(int)
seg_df['completed_flag'] = seg_df['offer completed'].notna().astype(int)

# # 0/1 플래그를 숫자형으로 정리
# seg_df['valid_attr_flag'] = seg_df['is_deduplicated'].astype(int)


In [39]:
gender_summary = seg_df.groupby('gender').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('completed_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum') # valid_attr_flag를 is_deduplicated로 바꾸기만 해도 ㄱㅊ을듯
).reset_index()

gender_summary['view_rate(%)'] = (
    gender_summary['viewed'] / gender_summary['total_offers'] * 100
).round(1)

gender_summary['view_drop_rate(%)'] = (
    100 - gender_summary['view_rate(%)']
).round(1)

gender_summary['completion_rate(%)'] = (
    gender_summary['completed'] / gender_summary['viewed'] * 100
).round(1)

gender_summary['completion_drop_rate(%)'] = (
    100 - gender_summary['completion_rate(%)']
).round(1)

display(
    gender_summary[
        [
            'gender',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,gender,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
1,M,30562,23012,16210,75.3,24.7,70.4,29.6,11063
0,F,21918,16876,15297,77.0,23.0,90.6,9.4,10018
2,O,721,612,493,84.9,15.1,80.6,19.4,363


In [40]:
seg_df['age_group'] = pd.cut(
    seg_df['age'],
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 120],
    labels=['10s', '20s', '30s', '40s', '50s', '60s', '70s', '80+'],
    right=False
)

seg_df['age_group'] = seg_df['age_group'].astype('object').fillna('Unknown')

age_summary = seg_df.groupby('age_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('completed_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum')
).reset_index()

age_summary['view_rate(%)'] = (
    age_summary['viewed'] / age_summary['total_offers'] * 100
).round(1)

age_summary['view_drop_rate(%)'] = (
    100 - age_summary['view_rate(%)']
).round(1)

age_summary['completion_rate(%)'] = (
    age_summary['completed'] / age_summary['viewed'] * 100
).round(1)

age_summary['completion_drop_rate(%)'] = (
    100 - age_summary['completion_rate(%)']
).round(1)

display(
    age_summary[
        [
            'age_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ]
)


,age_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
0,10s,722,512,315,70.9,29.1,61.5,38.5,201
1,20s,4997,3512,2347,70.3,29.7,66.8,33.2,1516
2,30s,5515,4039,2960,73.2,26.8,73.3,26.7,2018
3,40s,8241,6561,4788,79.6,20.4,73.0,27.0,3449
4,50s,12692,9704,8161,76.5,23.5,84.1,15.9,5387
5,60s,10780,8330,6826,77.3,22.7,81.9,18.1,4514
6,70s,6335,4820,4007,76.1,23.9,83.1,16.9,2645
7,80+,11760,9416,3697,80.1,19.9,39.3,60.7,2603


In [41]:
seg_df['income_group'] = pd.cut(
    seg_df['income'],
    bins=[0, 40000, 60000, 80000, 100000, float('inf')],
    labels=['0-40k', '40-60k', '60-80k', '80-100k', '100k+'],
    right=False
)

seg_df['income_group'] = seg_df['income_group'].astype('object').fillna('Unknown')

income_summary = seg_df.groupby('income_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('completed_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum')
).reset_index()

income_summary['view_rate(%)'] = (
    income_summary['viewed'] / income_summary['total_offers'] * 100
).round(1)

income_summary['view_drop_rate(%)'] = (
    100 - income_summary['view_rate(%)']
).round(1)

income_summary['completion_rate(%)'] = (
    income_summary['completed'] / income_summary['viewed'] * 100
).round(1)

income_summary['completion_drop_rate(%)'] = (
    100 - income_summary['completion_rate(%)']
).round(1)

display(
    income_summary[
        [
            'income_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,income_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
3,60-80k,16712,13084,10540,78.3,21.7,80.6,19.4,7200
2,40-60k,16101,12083,8384,75.0,25.0,69.4,30.6,5791
4,80-100k,9364,7645,7051,81.6,18.4,92.2,7.8,4652
5,Unknown,7841,6394,1101,81.5,18.5,17.2,82.8,889
0,0-40k,7072,4941,3001,69.9,30.1,60.7,39.3,2016
1,100k+,3952,2747,3024,69.5,30.5,110.1,-10.1,1785


In [42]:
# 정상 퍼널에 대해서 보려면 아래처럼 봐야함.
# seg_df['valid_funnel_flag'] = (
#     seg_df['offer viewed'].notna() &
#     seg_df['offer completed'].notna() &
#     (seg_df['offer viewed'] <= seg_df['offer completed'])
# ).astype(int)

#


---

In [45]:
# 정상케이스
seg_df['valid_funnel_flag'] = (
    seg_df['offer viewed'].notna() &
    seg_df['offer completed'].notna() &
    (seg_df['offer viewed'] <= seg_df['offer completed'])
).astype(int)


rows = []
for offer in seg_df['offer_id'].unique():
    temp = seg_df[seg_df['offer_id'] == offer]
    
    step1_received = len(temp)
    step2_viewed = temp['offer viewed'].notna().sum()
    step3_completed = temp['valid_funnel_flag'].sum()
    
    # 전환율 & 이탈률 계산
    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate
    
    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate
    
    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0
    
    rows.append({
        'offer_name': offer,
        'received': step1_received,
        'viewed': step2_viewed,
        'completed': step3_completed,
        'view_rate(%)': round(view_rate, 1),
        'view_drop_rate(%)': round(view_drop_rate, 1),
        'purchase_rate(%)': round(purchase_rate, 1),
        'purchase_drop_rate(%)': round(purchase_drop_rate, 1), 
        'total_cvr(%)': round(total_cvr, 1),
    })

offer_funnel_df = pd.DataFrame(rows)
display(offer_funnel_df.sort_values('total_cvr(%)', ascending=False))

,offer_name,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
1,discount_10_2_10,7597,7327,4576,96.4,3.6,62.5,37.5,60.2
3,discount_7_3_7,7646,7337,4348,96.0,4.0,59.3,40.7,56.9
0,bogo_5_5_5,7571,7264,3514,95.9,4.1,48.4,51.6,46.4
6,bogo_10_10_5,7593,7298,2739,96.1,3.9,37.5,62.5,36.1
7,bogo_10_10_7,7658,6716,2582,87.7,12.3,38.4,61.6,33.7
2,discount_10_2_7,7632,4118,2102,54.0,46.0,51.0,49.0,27.5
4,bogo_5_5_7,7677,4171,2106,54.3,45.7,50.5,49.5,27.4
5,discount_20_5_10,7668,2663,1300,34.7,65.3,48.8,51.2,17.0


In [ ]:
seg_df['valid_funnel_flag'] = (
    seg_df['offer viewed'].notna() &
    seg_df['offer completed'].notna() &
    (seg_df['offer viewed'] <= seg_df['offer completed'])
).astype(int)

def get_demographic_funnel(df, column_name):
    rows = []
    
    # 해당 컬럼(예: gender)에 있는 고유한 값들(M, F, O 등)을 하나씩 꺼내서 계산
    for group in df[column_name].dropna().unique():
        temp = df[df[column_name] == group]
        
        step1_received = len(temp)
        step2_viewed = temp['offer viewed'].notna().sum()
        step3_completed = temp['valid_funnel_flag'].sum() # 여기서 우연한 결제 제외!
        
        # 0으로 나누는 에러 방지
        view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
        purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
        total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0
        
        rows.append({
            column_name: group, # 어떤 그룹인지 이름표 붙이기
            'received': step1_received,
            'viewed': step2_viewed,
            'completed': step3_completed,
            'view_rate(%)': round(view_rate, 1),
            'view_drop_rate(%)': round(100 - view_rate, 1),
            'purchase_rate(%)': round(purchase_rate, 1),
            'purchase_drop_rate(%)': round(100 - purchase_rate, 1),
            'total_cvr(%)': round(total_cvr, 1),
        })
        
    result_df = pd.DataFrame(rows)
    # 결과를 보기 좋게 그룹 이름 순서대로 정렬해서 반환
    return result_df.sort_values(column_name).reset_index(drop=True)


print("[1] 성별(Gender) 퍼널 분석 결과")
gender_funnel = get_demographic_funnel(seg_df, 'gender')
display(gender_funnel)

print("\n[2] 연령대(Age Group) 퍼널 분석 결과")
age_funnel = get_demographic_funnel(seg_df, 'age_group')
display(age_funnel)

print("\n[3] 소득수준(Income Group) 퍼널 분석 결과")
income_funnel = get_demographic_funnel(seg_df, 'income_group')
display(income_funnel)

📊 [1] 성별(Gender) 퍼널 분석 결과


,gender,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
0,F,21918,16876,10426,77.0,23.0,61.8,38.2,47.6
1,M,30562,23012,11522,75.3,24.7,50.1,49.9,37.7
2,O,721,612,386,84.9,15.1,63.1,36.9,53.5



📊 [2] 연령대(Age Group) 퍼널 분석 결과


,age_group,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
0,10s,722,512,204,70.9,29.1,39.8,60.2,28.3
1,20s,4997,3512,1557,70.3,29.7,44.3,55.7,31.2
2,30s,5515,4039,2093,73.2,26.8,51.8,48.2,38.0
3,40s,8241,6561,3600,79.6,20.4,54.9,45.1,43.7
4,50s,12692,9704,5629,76.5,23.5,58.0,42.0,44.4
5,60s,10780,8330,4710,77.3,22.7,56.5,43.5,43.7
6,70s,6335,4820,2751,76.1,23.9,57.1,42.9,43.4
7,80+,11760,9416,2723,80.1,19.9,28.9,71.1,23.2



📊 [3] 소득수준(Income Group) 퍼널 분석 결과


,income_group,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
0,0-40k,7072,4941,2076,69.9,30.1,42.0,58.0,29.4
1,100k+,3952,2747,1828,69.5,30.5,66.5,33.5,46.3
2,40-60k,16101,12083,6000,75.0,25.0,49.7,50.3,37.3
3,60-80k,16712,13084,7530,78.3,21.7,57.6,42.4,45.1
4,80-100k,9364,7645,4900,81.6,18.4,64.1,35.9,52.3
5,Unknown,7841,6394,933,81.5,18.5,14.6,85.4,11.9
